In [22]:
LLM_PROVIDER = "groq"          #groq / gemini / openai
LLM_MODEL = "openai/gpt-oss-20b"   #change model here based on your use-case.
CORPUS_PATH = "./knowledge_base/"
print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")

Provider: groq
Model: openai/gpt-oss-20b


In [23]:
%pip install streamlit python-dotenv langchain-community langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import os
from dotenv import load_dotenv
load_dotenv()
if os.getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY loaded from .env")
else:
    print("WARNING: GROQ_API_KEY not found.")

GROQ_API_KEY loaded from .env


## Importing all the documents
Using PyPDFDirectoryLoader to Load all the PDF Files

In [25]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(CORPUS_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents") #len = number of pages in the combined document

Loaded 39 documents


## Splitting the combined text into Chunks

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# chunk_size: target characters per chunk
# chunk_overlap: characters shared between consecutive chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 107 chunks


## Initiallizing Embedding Model

In [27]:
from langchain_huggingface import HuggingFaceEmbeddings
# A lightweight, widely-used sentence embedding model — runs comfortably on CPU
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model initialized.


## Vector Store Initiallization

In [28]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store initialized.")

Vector store initialized.


## Initializing LLM

In [29]:
from langchain_groq import ChatGroq
llm = ChatGroq(
        model=LLM_MODEL,
        temperature=0.7,
        max_tokens=512
    )

## Build the RAG Chain


In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an HR assistant. Answer the question using ONLY
the context below. If the answer isn't in the context,
say you don't have that information.

Context: {context}
Question: {question}""")


def format_docs(docs):
    # Join retrieved chunks into one context string
    return "\n\n".join(d.page_content for d in docs)


@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = RAG_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    return {"answer": answer, "sources": docs}


print("RAG pipeline initialized.")


RAG pipeline initialized.


## Guardials
To answer only the questions within the scope of the the Chatbot

In [31]:
GUARDRAIL_PROMPT = ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant. Decide whether the question
below is something an HR assistant should answer (company leave policy,
reimbursement, code of conduct, or other internal HR topics) or something
out of scope (general knowledge, coding help, unrelated small talk, etc).
Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.
Question: {question}
""")
REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)
@traceable(name="ask_bot")
def ask_bot(question: str):
    guardrail_chain = GUARDRAIL_PROMPT | llm | StrOutputParser()
    verdict = guardrail_chain.invoke({"question": question}).strip().upper()

    if "OUT_OF_SCOPE" in verdict:
        return {"answer": REFUSAL_MESSAGE, "sources": []}

    return rag_chain(question)

print("Guardrails initialized.")


Guardrails initialized.


In [32]:
test_questions = [
    "How many casual leaves do I get per year?",
    "What is the internet reimbursement limit?",
    "What's the capital of France?",
]
for i, q in enumerate(test_questions, 1):
    result = ask_bot(q)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}")
    if result["sources"]:
        print(
            f"Sources: "
            f"{[d.metadata.get('source') for d in result['sources']]}"
        )
    print("-" * 60)

Q1: How many casual leaves do I get per year?
A1: I don’t have that information.
Sources: ['knowledge_base/06_Compensation_and_Benefits_Policy.pdf', 'knowledge_base/06_Compensation_and_Benefits_Policy.pdf', 'knowledge_base/02_Leave_Policy.pdf']
------------------------------------------------------------
Q2: What is the internet reimbursement limit?
A2: The internet reimbursement limit is **Rs. 1,000 per month** for eligible employees.
Sources: ['knowledge_base/03_Work_From_Home_Policy.pdf', 'knowledge_base/07_IT_and_Data_Security_Policy.pdf', 'knowledge_base/10_Travel_and_Expense_Policy.pdf']
------------------------------------------------------------
Q3: What's the capital of France?
A3: I'm an HR assistant and can only help with questions about company HR policies (leave, reimbursement, code of conduct, etc.). I don't have information to answer that question.
------------------------------------------------------------


## Sample Solution

In [33]:
import pandas as pd
test_df = pd.read_csv("test.csv")
answers = []
for i, row in test_df.iterrows():
    q = row["question"]
    result = ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/{len(test_df)}] {q}\n-> {result['answer']}\n{'-'*60}")
test_df["answer"] = answers
submission = test_df[["question_id", "answer"]]
submission.to_csv("submission.csv", index=False)
submission.head()

[1/20] How does my Earned Leave accrue every month?
-> Earned Leave (EL) accrues at a rate of **1.25 days per month** after you have completed one year of continuous service.  
If you are still in your probation period, EL accrues at **0.5 days per month** and those days become available for use only after probation confirmation.
------------------------------------------------------------
[2/20] How much Earned Leave can I carry forward to next year?
-> I don’t have that information.
------------------------------------------------------------
[3/20] How many weeks of Maternity Leave am I entitled to?
-> According to the company policy, a female employee who meets the service requirement is entitled to:

* **26 weeks of paid Maternity Leave** for the first and second live births.  
* **12 weeks of paid Maternity Leave** for a third child.  
* Up to **8 weeks of pre‑natal leave** may be taken before the expected delivery date.

So your entitlement depends on which birth you are having—

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01k0mm8gpee2tvrcnb1y85qknp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6899, Requested 1319. Please try again in 1.635s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}